In [ ]:
/**
 * ================================================================================
 *   FANOUT PATTERN — EXPLAINED LIKE WE'RE JUST TALKING ABOUT IT
 * ================================================================================
 * 
 *   Okay so imagine this. You post "Hello world" on a social app.
 *   Saving your post? Easy. One write to a database. Done in milliseconds.
 * 
 *   But here's the real question nobody asks out loud:
 * 
 *       "Okay cool, the post is saved... but how does it actually SHOW UP
 *        in the feeds of your 1,000 followers?"
 * 
 *   That "how do we get this one thing to show up for a LOT of people"
 *   problem is called FANOUT.
 * 
 *   Simple definition:
 *     Fanout = one event happens, and it needs to turn into MANY updates
 *     (or many things to check) somewhere else.
 * 
 *   Where the word comes from:
 *     It's borrowed from electronics. In a circuit, "fan-out" means how
 *     many other components one output pin can drive/power. Same idea
 *     here — one action "fans out" into many downstream effects.
 * 
 *   Real world analogy:
 *     Think of a teacher announcing "no homework tonight!" to the class.
 *     - The teacher only says it ONCE (that's the "write")
 *     - But NOW every single student needs to know about it (that's the "fanout")
 * 
 *     The teacher has two choices:
 *       (a) Personally walk to each of the 30 students' desks and tell them
 *           individually right now (push it out immediately)
 *       (b) Just say it once out loud, and each student figures out they
 *           heard it whenever they're paying attention (pull it when needed)
 * 
 *   That's literally the whole fanout pattern. Two choices, everything else
 *   is details.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   1. WHY DOES THIS EVEN MATTER? (Where you'll see this pattern)
 * --------------------------------------------------------------------------------
 * 
 *   This shows up ANY time "one thing" needs to become "many things":
 * 
 *     Problem                  |  Why fanout shows up
 *     --------------------------|--------------------------------------------
 *     Twitter / Instagram       |  One post -> shows up in millions of feeds
 *     Notification systems      |  One event -> alerts to many subscribed users
 *     News feed aggregation     |  Combine posts from everyone you follow
 *     Activity feeds            |  Show actions from your connections
 *     Reddit / forums           |  One new post -> shown to all subscribers
 * 
 *   The core question in ALL of these:
 * 
 *       "When one user creates something, how do we get it to the right
 *        people WITHOUT melting our servers?"
 * 
 * 
 * --------------------------------------------------------------------------------
 *   2. THE BIG PICTURE DIAGRAM
 * --------------------------------------------------------------------------------
 * 
 *   Here's what fanout looks like visually. One post, many destinations:
 * 
 *                               ┌─────────────────┐
 *                               │   User Posts     │
 *                               │  "Hello world"   │
 *                               └────────┬─────────┘
 *                                        │
 *                                        ▼
 *                               ┌─────────────────┐
 *                               │  Fanout Service  │
 *                               └────────┬─────────┘
 *                      ┌─────────────────┼─────────────────┐
 *                      │                 │                 │        ...
 *                      ▼                 ▼                 ▼
 *               ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
 *               │ Follower 1  │   │ Follower 2  │   │ Follower N  │
 *               │   Feed      │   │   Feed      │   │   Feed      │
 *               └─────────────┘   └─────────────┘   └─────────────┘
 * 
 *   If that user has 1,000 followers, ONE post can turn into 1,000 updates.
 *   That ratio (1 post -> 1,000 updates) is called the "fanout ratio."
 * 
 *   Now imagine that user has 50 MILLION followers. Suddenly "one post"
 *   means fifty million updates. THAT'S the scary part of fanout, and it's
 *   the reason this whole topic exists as a system design problem.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   3. QUESTIONS TO ASK BEFORE YOU DESIGN ANYTHING
 * --------------------------------------------------------------------------------
 * 
 *   Before picking an approach, you need to understand the "shape" of the
 *   problem. Think of it like a doctor asking questions before prescribing
 *   anything — don't just jump to a solution.
 * 
 *     Question                                  | Why it matters
 *     --------------------------------------------|------------------------------
 *     How many followers does a typical user have?| Tells you write cost
 *     How many people does a typical user follow?  | Tells you read cost
 *     Is the feed chronological or ranked?         | Ranked = needs scoring too
 *     How fresh does the feed need to be?          | Can delays be tolerated?
 *     Is losing a feed entry a big deal, or is it  | Tells you how much you need
 *       recoverable/recomputable?                  |   retries & cleanup logic
 *     What happens on delete/block/mute/privacy?   | Feed cache isn't "truth" —
 *                                                   |   you still need to filter
 * 
 *   Saying "I'll just use fanout-on-write" without answering these is like
 *   a contractor saying "I'll just build a house" without asking how many
 *   people live there or what the budget is.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   4. APPROACH #1 — FANOUT-ON-WRITE (a.k.a. "PUSH")
 * --------------------------------------------------------------------------------
 * 
 *   ANALOGY: This is the teacher who personally walks around and tells
 *   every single student the news, right when it happens. A lot of work
 *   up front, but every student already knows by the time they need it.
 * 
 *   HOW IT WORKS:
 *     1. User posts something
 *     2. We save the post (1 write)
 *     3. We look up ALL of that user's followers
 *     4. We push a copy/reference of that post into EACH follower's
 *        personal feed cache
 *     5. Later, when a follower opens the app, their feed is basically
 *        already sitting there waiting for them — just read and go.
 * 
 *   DIAGRAM (sequence of events):
 * 
 *     User A        Post Service     Fanout Service      Feed Cache        Follower
 *       │                │                  │                 │               │
 *       │──Create Post──▶│                  │                 │               │
 *       │                │──Store Post─────▶│                 │               │
 *       │                │──Enqueue Fanout─▶│                 │               │
 *       │                │                  │──Get Followers─▶│               │
 *       │                │                  │──Write Follower1 Feed──────────▶│
 *       │                │                  │──Write Follower2 Feed──────────▶│
 *       │                │                  │        ... (async, in background)
 *       │                │                                                    │
 *       │                │                                    │──Read Feed───▶│
 *       │                │                                    │◀─Cached data──│
 * 
 *   Notice: the fanout writes happen in the BACKGROUND (async), after the
 *   user already got "post successful!" This matters a lot for speed.
 * 
 *   DATA STRUCTURE EXAMPLE (this is what a feed cache looks like, e.g. Redis):
 * 
 *     feed:userB  ->  [ post_123 (time: 1703001000),
 *                        post_456 (time: 1703000500),
 *                        post_789 (time: 1703000000) ]
 * 
 *     feed:userC  ->  [ post_123 (time: 1703001000),
 *                        post_450 (time: 1703000800) ]
 * 
 *   It's basically a sorted list per user, newest post first. Redis "sorted
 *   sets" are great for this because inserting and reading a range is fast
 *   (O(log N), if you're familiar with that notation — just means "fast
 *   even as it grows").
 * 
 *   PROS (why people like this):
 *     ✔ Reading your feed is FAST — it's basically pre-made, just serve it
 *     ✔ Read speed doesn't change no matter how many people you follow
 *     ✔ Simple code when a user opens their feed — no heavy computing
 *     ✔ Great when people check their feed way more often than they post
 * 
 *   CONS (the catch):
 *     ✘ Tons of background write work per post
 *     ✘ You compute feeds for people who might not even log in — wasted effort
 *     ✘ Same post gets duplicated across millions of feed caches — storage cost
 *     ✘ "Fanout lag" — if the background workers are behind, followers see
 *       the post late
 * 
 *   THE BIG PROBLEM — the "celebrity" problem:
 * 
 *     ┌───────────────────┐
 *     │  Author posts,     │
 *     │  has 50,000,000    │
 *     │  followers         │
 *     └─────────┬──────────┘
 *               ▼
 *      ┌──────────────────┐
 *      │  Fanout Service   │
 *      └─────────┬─────────┘
 *         ┌───────┼────────────────────────┐
 *         ▼       ▼                        ▼
 *      Write 1  Write 2   ...   Write 50,000,000  😱
 * 
 *   If a celebrity with 50 million followers posts, and you push it to
 *   EVERY follower's feed cache instantly, you just created 50 million
 *   write jobs from ONE post. That can jam up your entire system and
 *   delay everyone else's posts too. This is exactly why the "pull"
 *   approach exists.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   5. APPROACH #2 — FANOUT-ON-READ (a.k.a. "PULL")
 * --------------------------------------------------------------------------------
 * 
 *   ANALOGY: This is the teacher who just says the announcement out loud
 *   ONCE. Nothing is pushed to anyone. Each student has to actively listen
 *   and "pull" the info whenever they check in. Way less work for the
 *   teacher, but each student has to do a bit of work to catch up.
 * 
 *   HOW IT WORKS:
 *     1. User posts something
 *     2. We save the post ONCE — that's it, no fanout happens yet
 *     3. Later, when a follower opens their feed, WE query all the
 *        people they follow, grab their recent posts, and MERGE + SORT
 *        them together, right at that moment.
 * 
 *   DIAGRAM (sequence of events):
 * 
 *     User A       Post Service    Follower       Feed Service      Posts DB
 *       │               │              │               │                │
 *       │──Create Post─▶│              │               │                │
 *       │               │──Store Post (single write)───────────────────▶│
 *       │               │◀─Post Created─                                │
 *       │                              │──Get My Feed─▶│                │
 *       │                              │               │──Get Following [A,B,C]
 *       │                              │               │──Query Posts from A,B,C▶│
 *       │                              │               │◀────results────────────│
 *       │                              │               │──Merge & Sort──│
 *       │                              │◀─Return Feed──│                │
 * 
 *   Notice: NOTHING happened at post time except one simple save. All the
 *   "work" of building the feed happens when the follower actually asks
 *   for it.
 * 
 *   QUERY PATTERN (rough idea, in plain English):
 *     "Get me the following list for this user, then go grab recent posts
 *     from each of those accounts, then merge them all together sorted
 *     by time." In practice, teams query a limited recent window per
 *     followed author (not their entire history) to keep this fast.
 * 
 *   PROS (why people like this):
 *     ✔ Writing a post is SUPER fast — just one save, no fanout needed
 *     ✔ Zero wasted work — you only build a feed for people who ask for one
 *     ✔ No celebrity-post-explosion problem at write time
 *     ✔ Way less storage — no duplicated post references everywhere
 * 
 *   CONS (the catch):
 *     ✘ Reading your feed is SLOWER — lots of querying + merging happens live
 *     ✘ The more accounts you follow, the slower your feed loads
 *     ✘ More complex code running on every single feed view
 *     ✘ Popular times of day (everyone opening the app) can hammer your database
 *     ✘ If the feed is "ranked" (not just newest-first), you STILL need to
 *       score everything at read time — extra cost
 * 
 * 
 * --------------------------------------------------------------------------------
 *   6. SIDE-BY-SIDE COMPARISON
 * --------------------------------------------------------------------------------
 * 
 *   Aspect                | Fanout-on-Write (Push)     | Fanout-on-Read (Pull)
 *   -----------------------|----------------------------|---------------------------
 *   Write speed            | Fast to acknowledge, but   | Fast (just 1 write)
 *                           |  heavy background work    |
 *   Read speed             | Fast (pre-built)           | Slower, more variable
 *   Storage cost           | High (duplicated refs)     | Low (stored once)
 *   Where the "cost" lives | At write time              | At read time
 *   Best for               | Read-heavy apps, normal    | Write-heavy apps, huge
 *                           |  follower counts           |  follower-count spread
 *   Celebrity problem?     | BIG issue (huge write job) | Not an issue at write time
 *   Freshness              | Can lag behind (async)     | Usually fresh (reads live data)
 * 
 *   Simple way to remember it:
 * 
 *       Fanout-on-write  = pay the cost NOW, up front, so reads are easy later
 *       Fanout-on-read   = pay the cost LATER, only when someone actually looks
 * 
 *   Neither one is "the right answer" on its own — they're a trade-off dial,
 *   not a multiple-choice question with one correct option.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   7. THE HYBRID APPROACH (what big companies actually do)
 * --------------------------------------------------------------------------------
 * 
 *   ANALOGY: Back to our teacher. What if the teacher personally tells
 *   most students (there aren't that many), BUT for the school's giant
 *   600-person assembly hall, instead of running around to 600 people,
 *   she just posts a note on the door that people check when they walk in?
 * 
 *   That's the hybrid model: treat "normal" people one way, and "huge
 *   audience" people another way.
 * 
 *   THE CORE IDEA:
 *     - Most users have a normal, manageable number of followers
 *       → fan out their posts eagerly (push), it's cheap and keeps reads fast
 *     - A tiny number of users are "celebrities" with millions of followers
 *       → DON'T push their posts everywhere. Just store once, and pull it
 *         in at read time, merging it into whoever's feed needs it.
 * 
 *   DECISION DIAGRAM:
 * 
 *                          ┌─────────────┐
 *                          │  New Post   │
 *                          └──────┬──────┘
 *                                 ▼
 *                     ┌───────────────────────┐
 *                     │ Is Author High-Follower?│
 *                     └─────────┬─────────┬────┘
 *                         No     │         │  Yes
 *                     ┌──────────┘         └───────────┐
 *                     ▼                                 ▼
 *           ┌───────────────────┐              ┌────────────────┐
 *           │  Fanout-on-Write   │              │  Store Only     │
 *           │ (push to followers)│              │ (in Posts DB)   │
 *           └──────────┬──────────┘              └────────┬────────┘
 *                      │                                    │
 *                      └──────────────┬─────────────────────┘
 *                                      ▼
 *                           ┌────────────────────┐
 *                           │  Feed Request       │
 *                           │  MERGE both sources │
 *                           └──────────┬──────────┘
 *                                      ▼
 *                               ┌─────────────┐
 *                               │ Final Feed  │
 *                               └─────────────┘
 * 
 *   WHAT "HIGH-FOLLOWER" MEANS IS UP TO YOU:
 *     There's no magic number. You tune it based on:
 *       - How much fanout work your system can actually handle
 *       - How often that author posts (frequent posters = lower threshold)
 *       - How many of their followers are ACTUALLY active (50K followers
 *         but only 1K active ≠ same cost as 50K all-active followers)
 * 
 *   A SIMPLE WAY TO TRACK THIS — add a flag per user:
 * 
 *     user_A  |  15,000,000 followers  |  eager_fanout = false
 *     user_B  |          500 followers |  eager_fanout = true
 *     user_C  |      50,000 followers  |  eager_fanout = false
 * 
 *   A background job can re-check this flag periodically as follower
 *   counts change, kind of like re-sorting people into "VIP" vs "regular"
 *   as their popularity changes.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   8. TRICKY EDGE CASES (the stuff interviews actually dig into)
 * --------------------------------------------------------------------------------
 * 
 *   a) A frequent, high-follower poster
 *      Even if you skip write-time fanout for them, reading THEIR posts
 *      at read-time repeatedly can still be expensive. Fixes: cache their
 *      recent posts, limit how many of their posts count as "candidates,"
 *      and separate feed delivery from push notifications.
 * 
 *   b) Someone suddenly goes viral (crosses the celebrity threshold)
 *      You don't need to rewrite history. Just apply the new rule to
 *      FUTURE posts, and let read-time merging smooth over the transition.
 * 
 *   c) You just followed someone new — do you see their old posts?
 *      - Fanout-on-read: automatic! The read path already looks them up.
 *      - Fanout-on-write: NOT automatic — your feed cache was built
 *        before you followed them. Fix: backfill a small recent window of
 *        their posts into your cache the moment you follow them.
 * 
 *   d) Someone deletes a post
 *      It might already be sitting in millions of feed caches. Typical
 *      fix:
 * 
 *          Delete Request → Mark Deleted in DB → Background Cleanup Job
 *                                                        │
 *                                                        ▼
 *                                           Remove from Feed Caches
 * 
 *      IMPORTANT: the read path should ALSO double check "is this post
 *      still visible?" as a safety net — you can't rely on the cleanup
 *      job finishing instantly.
 * 
 *   e) Blocks, mutes, privacy changes
 *      Same idea — the feed cache is NOT the final word on what you're
 *      allowed to see. Always double-check visibility rules when actually
 *      returned a feed, regardless of what's cached.
 * 
 *   f) Ranking (not just chronological)
 *      Fanout gives you CANDIDATES, not necessarily the final order. You
 *      may still need to score them (freshness, relationship strength,
 *      engagement, etc.) at read time.
 * 
 *   g) "I just posted and I don't see my own post!"
 *      This is called the "read-your-own-writes" problem. Fixes:
 *      always include your own recent posts directly in your feed
 *      response, or optimistically show it on your device before the
 *      backend even confirms.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   9. MAKING IT FAST IN PRACTICE (performance tricks)
 * --------------------------------------------------------------------------------
 * 
 *   BATCHING
 *     Don't write to 10,000 follower caches one at a time — that's 10,000
 *     separate trips. Group them into batches and use bulk writes.
 * 
 *   ASYNC PROCESSING (the big one)
 *     Don't make the user wait for fanout to finish before saying "posted!"
 * 
 *          Post Service ──▶ Message Queue ──▶ Fanout Worker 1 ─┐
 *                                           ├─▶ Fanout Worker 2 ─┼─▶ Feed Cache
 *                                           └─▶ Fanout Worker N ─┘
 * 
 *     The post service just drops a job on a queue and immediately
 *     responds to the user. Workers process the queue in the background.
 *     Bonus: if a worker fails, it can just retry the job.
 * 
 *     IMPORTANT: workers must be "idempotent" — meaning if a job
 *     accidentally runs twice (due to a retry), it should NOT create
 *     duplicate feed entries. Usually solved by de-duplicating on
 *     (user_id, post_id) — like a bouncer checking a guest list before
 *     letting someone in twice.
 * 
 *   MULTI-LAYER CACHING
 *     Layer          | What it stores                | Common Tech
 *     ----------------|-------------------------------|---------------------
 *     Feed cache      | Precomputed feed entries       | Redis Sorted Sets
 *     Post cache      | Full post content (hydrated)   | Redis / Memcached
 *     User cache      | Follower lists, profile data   | Redis
 *     CDN             | Images, static files           | Cloudflare, CloudFront
 * 
 *   KEEP FEED CACHES SMALL
 *     You don't need to store someone's ENTIRE post history in every
 *     follower's cache. A common rule: keep only the last ~800 posts, or
 *     posts from the last 7 days. Older stuff can be looked up from the
 *     main database on the rare occasion someone scrolls back that far.
 * 
 *   BACKPRESSURE (what to do when things get overloaded)
 *     If your fanout queue starts piling up, don't let it take down the
 *     whole app. Options: slow down low-priority fanout, delay
 *     notifications, temporarily rely more on read-time pulling. Protect
 *     "core" actions (posting, reading a feed) FIRST, let background
 *     fanout catch up when it can.
 * 
 * 
 * --------------------------------------------------------------------------------
 *   10. QUICK-REFERENCE: WHICH APPROACH SHOULD I USE?
 * --------------------------------------------------------------------------------
 * 
 *   Use FANOUT-ON-WRITE when:
 *     [ ] Reads happen WAY more often than writes (typical social feed)
 *     [ ] Most users have a "normal," fairly similar follower count
 *     [ ] You can tolerate a small delay before followers see new posts
 *     [ ] You want simple, fast, predictable read-time code
 * 
 *   Use FANOUT-ON-READ when:
 *     [ ] Writes need to be instant and cheap no matter what
 *     [ ] Follower counts vary WILDLY (some tiny, some huge)
 *     [ ] Freshness matters more than raw read speed
 *     [ ] You want to avoid wasting effort on users who rarely log in
 * 
 *   Use the HYBRID approach when:
 *     [ ] You have both "everyday users" AND "celebrity-scale" accounts
 *     [ ] Write amplification from top accounts is becoming a real problem
 *     [ ] You're building something at real production scale (Twitter-like)
 *     [ ] You're okay maintaining a bit more complexity for a lot more efficiency
 * 
 *   Whichever you pick, ALWAYS also ask yourself:
 *     [ ] What happens on delete / block / mute / privacy change?
 *         (Feed caches are hints, not the final truth — always re-check
 *          visibility on read)
 *     [ ] Are my background workers safe to retry without creating
 *         duplicate entries? (idempotency)
 *     [ ] Do I have a plan for what happens when fanout falls behind
 *         (backpressure), instead of just hoping it never happens?
 *     [ ] Will a new follow show old posts, or only new ones going forward?
 *         Have I decided on purpose, instead of by accident?
 * 
 * 
 * --------------------------------------------------------------------------------
 *   TL;DR — THE ONE THING TO REMEMBER
 * --------------------------------------------------------------------------------
 * 
 *   Fanout is really just one question in disguise:
 * 
 *       "WHEN do I pay the cost of getting one thing to many people —
 *        right when it's created, right when someone asks for it, or
 *        some smart mix of both?"
 * 
 *   Everything else in this topic (caching, batching, queues, ranking,
 *   edge cases) is just the details of making whichever choice you pick
 *   actually work well at scale.
 * 
 * ================================================================================
*/

In [ ]:
/**
 * ====================================================================
 * ROUND 1: PROBLEM SAMAJHNA (Understanding the Problem)
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Chalo, design karo ek system jaisa Twitter/Instagram ka feed hota
 *    hai. Jab koi user post karta hai, uske followers ko woh post
 *    dikhna chahiye. Kaise design karoge?"
 * 
 * CANDIDATE:
 *   "Sir, is problem mein sabse important cheez yeh decide karna hai
 *    ki hum FANOUT kab karenge - post likhte waqt (write time) ya
 *    feed padhte waqt (read time). Usse pehle main kuch clarifying
 *    questions poochna chahunga."
 * 
 *   [YAHAN CANDIDATE SMART DIKHTA HAI - SEEDHA SOLUTION PE MAT KUDO]
 * 
 * INTERVIEWER:
 *   "Theek hai, pucho."
 * 
 * CANDIDATE (Clarifying Questions - yeh bolna zaroori hai):
 *   1. "Average user ke kitne followers hote hain? Aur kya kuch
 *       users ke MILLIONS of followers ho sakte hain (celebrities)?"
 *   
 *   2. "Average user kitne logo ko follow karta hai?"
 *   
 *   3. "Feed CHRONOLOGICAL chahiye (time-wise) ya RANKED/algorithmic
 *       chahiye (jaise engagement ke basis pe)?"
 *   
 *   4. "Feed kitni FRESH honi chahiye? Real-time turant, ya kuch
 *       seconds/minutes ka delay chalega?"
 *   
 *   5. "Scale kitna hai? Kitne DAU (daily active users) expect
 *       karte hain?"
 *   
 *   6. "Delete, block, mute, private account jaisi features
 *       support karni hain?"
 * 
 * INTERVIEWER:
 *   "Achha sawaal. Maan lo: average user ke 200 followers hain,
 *    lekin kuch celebrities ke 50 million tak followers hain.
 *    Feed thoda ranked hona chahiye, aur 1-2 second ka delay
 *    chalega. Scale hai 100 million users."
 * 
 * CANDIDATE:
 *   "Perfect, is information se clear hai ki:
 *      - Follower distribution BAHUT UNEVEN hai (normal users vs
 *        celebrities) - iska matlab hum SIRF ek approach (pure push
 *        ya pure pull) use nahi kar sakte.
 *      - Thoda delay chalta hai, toh ASYNC processing use kar sakte
 *        hain.
 *      - Ranked feed hai, toh candidate generation + scoring dono
 *        chahiye honge.
 *    Main pehle dono basic approaches explain karta hoon, phir
 *    bataunga hum kya choose karenge is case mein."
 * 
 *   [YEH LINE BOLNA IMPORTANT HAI - Interviewer ko dikhta hai ki tum
 *    STRUCTURED soch rahe ho, seedha jump nahi kar rahe]
*/

/**
 * ====================================================================
 * ROUND 2: FANOUT-ON-WRITE (PUSH MODEL) - Deep Dive
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Chalo pehle bata do, agar hum simple approach lein jahan post
 *    hote hi sabke feed mein daal dein, toh kaise design karoge?"
 * 
 * CANDIDATE:
 *   "Yeh 'Fanout-on-Write' ya 'Push Model' hai. Idea simple hai:
 *    jab post create hota hai, hum turant (background mein) uske
 *    har follower ki feed cache mein woh post ka reference likh dete
 *    hain. Jab follower feed kholta hai, usko already-ready data
 *    milta hai."
 * 
 *   DIAGRAM (bolke draw kar sakte ho whiteboard pe):
 * 
 *     User A --post--> [Post Service] --save--> [Posts DB]
 *                             |
 *                             v
 *                     [Enqueue Fanout Job]   (Message Queue mein)
 *                             |
 *                             v
 *                     [Fanout Worker(s)]
 *                             |
 *                 Get Followers List [F1, F2, ... FN]
 *                             |
 *           -----------------------------------------
 *           |                    |                  |
 *           v                    v                  v
 *     Write to F1 Feed    Write to F2 Feed   Write to FN Feed
 *        (Redis)              (Redis)            (Redis)
 * 
 * INTERVIEWER:
 *   "Feed cache kaise store karoge? Data structure batao."
 * 
 * CANDIDATE:
 *   "Redis Sorted Set best fit hai. Har user ka apna sorted set
 *    hoga, jisme post_id key hai aur timestamp score hai:
 * 
 *      feed:userB
 *        post_123  -> score 1703001000  (sabse naya - top pe)
 *        post_456  -> score 1703000500
 *        post_789  -> score 1703000000  (sabse purana)
 * 
 *    Sorted Set isliye kyunki:
 *      - Insert O(log N) mein hota hai
 *      - 'Latest N posts do' jaisi range query bahut fast hai
 *    
 *    Aur main har feed cache ko CAP kar dunga - jaise sirf last 800
 *    posts ya last 7 din ka data rakhunga, taaki memory bounded rahe."
 * 
 * INTERVIEWER:
 *   "Iske fayde aur nuksaan kya hain?"
 * 
 * CANDIDATE:
 *   "FAYDE:
 *      - Read bahut fast hai - already precomputed hai, seedha cache
 *        se utha lo
 *      - Read latency predictable rehti hai chahe kitna bhi following
 *        ho
 *      - Read-heavy systems ke liye perfect (log baar baar refresh
 *        karte hain, kam post karte hain)
 * 
 *    NUKSAAN:
 *      - Write time pe bahut zyada kaam - agar celebrity post kare
 *        jiske 50 million followers hain, toh 50 million writes!
 *      - Wasted work - un logo ke liye bhi likhte hain jo kabhi login
 *        hi nahi karenge
 *      - Storage zyada - same post_id lakho jagah duplicate hota hai
 *      - Fanout lag - worker ko time lagta hai, follower thodi der
 *        baad dekhega post"
 * 
 * INTERVIEWER:
 *   "Yeh 50 million writes wala case hi problem hai na? Isko
 *    'celebrity problem' bolte hain. Iske baare mein bolo."
 * 
 * CANDIDATE:
 *   "Bilkul sir, isko HIGH-FOLLOWER AUTHOR PROBLEM kehte hain.
 * 
 *     DIAGRAM:
 * 
 *       [Celebrity Posts] (50M Followers)
 *               |
 *               v
 *         [Fanout Service]
 *               |
 *       Write 1, Write 2, ... Write 50,000,000
 *               |
 *               v
 *       [Queue Backlog / Worker Overload]
 *               |
 *               v
 *       Doosre normal users ke posts bhi DELAY ho jayenge!
 * 
 *    Yeh ek single celebrity ka post poore system ke fanout workers
 *    ko busy kar sakta hai, jisse baaki sab logo ka fanout bhi slow
 *    ho jata hai. Isi wajah se pure push-model bade scale pe akela
 *    kaam nahi karta - isko fix karne ke liye hum HYBRID approach
 *    use karte hain, jo main aage explain karunga."
*/

/**
 * ====================================================================
 * ROUND 3: FANOUT-ON-READ (PULL MODEL) - Deep Dive
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Theek hai, ab doosra approach batao jahan hum kuch bhi precompute
 *    nahi karte."
 * 
 * CANDIDATE:
 *   "Yeh 'Fanout-on-Read' ya 'Pull Model' hai. Idea hai: post sirf EK
 *    BAAR database mein save hota hai. Koi follower cache update nahi
 *    hoti. Jab follower feed maangta hai, TAB system uske following
 *    list ke sab authors ke recent posts nikaal ke, merge karke,
 *    sort karke deta hai."
 * 
 *   DIAGRAM (Write path - bahut simple):
 * 
 *     User A --post--> [Post Service] --single save--> [Posts DB]
 *     (Bas itna hi! Koi fanout nahi hota abhi)
 * 
 *   DIAGRAM (Read path - asli kaam yahan):
 * 
 *     Follower1 --"feed do"--> [Feed Service]
 *                                     |
 *                         Get Following List [A, B, C]
 *                                     |
 *                 -----------------------------------------
 *                 |                  |                    |
 *                 v                  v                    v
 *         Query A's posts    Query B's posts      Query C's posts
 *                 |                  |                    |
 *                 -----------------------------------------
 *                                    |
 *                                    v
 *                           [Merge & Sort by time]
 *                                    |
 *                                    v
 *                           Return Top 100 Posts
 * 
 * INTERVIEWER:
 *   "Isme problem kya aa sakti hai agar user 5000 logo ko follow
 *    karta ho?"
 * 
 * CANDIDATE:
 *   "Bahut badi problem hai - agar user 5000 accounts follow karta
 *    hai, toh feed request pe potentially 5000 alag queries chalani
 *    pad sakti hain (ya kam se kam bahut sources se data lana padega),
 *    phir sabko merge karna padega. Yeh READ LATENCY ko slow aur
 *    UNPREDICTABLE bana deta hai.
 * 
 *    Isliye practically hum:
 *      - Per-author timeline maintain karte hain
 *      - Har source se sirf BOUNDED recent window query karte hain
 *        (jaise last 20 posts har author se)
 *      - Phir un candidates ko merge karte hain
 * 
 *    Yeh approach query cost ko control mein rakhta hai."
 * 
 * INTERVIEWER:
 *   "Fayde aur nuksaan?"
 * 
 * CANDIDATE:
 *   "FAYDE:
 *      - Write bahut fast - sirf ek DB entry
 *      - Wasted work nahi hota - sirf active users ke liye compute
 *        hota hai
 *      - Celebrity/high-follower author koi problem create nahi
 *        karta write time pe
 *      - Storage kam lagta hai
 * 
 *    NUKSAAN:
 *      - Read slow ho sakti hai
 *      - Read latency variable hai - jyada following = jyada slow
 *      - Read-side code complex hai (merge + scoring logic)
 *      - Peak hours mein DB pe load zyada aa sakta hai
 *      - Agar RANKED feed chahiye, toh scoring bhi read time pe
 *        karni padegi - aur bhi zyada slow"
 * 
 * 
 * ====================================================================
 * ROUND 4: COMPARISON - Interviewer Confuse Karne Ki Koshish Karega
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Toh dono mein se better kaunsa hai? Bas ek batao."
 * 
 * CANDIDATE (YEH TRAP HAI - IDEAL ANSWER YEH NAHI HAI KI EK CHUN LO):
 *   "Sir, dono ka apna trade-off hai, koi ek universally better nahi
 *    hai. Yeh depend karta hai workload pe:
 * 
 *      - Agar system READ-HEAVY hai aur follower count sabka roughly
 *        uniform hai -> Fanout-on-Write better hai
 *      
 *      - Agar system WRITE-HEAVY hai ya follower count mein bahut
 *        variance hai (kuch celebrities, baaki normal) -> Fanout-on-
 *        Read better hai un high-follower cases ke liye
 * 
 *    Real production systems (jaise Twitter) DONO ko combine karte
 *    hain - isko HYBRID APPROACH kehte hain, jo main next explain
 *    karta hoon."
 * 
 *   QUICK COMPARISON TABLE (yaad rakhne ke liye):
 * 
 *     Aspect          | Fanout-on-Write   | Fanout-on-Read
 *     ----------------|-------------------|------------------
 *     Write Speed     | Slow              | Fast
 *     Read Speed      | Fast              | Slow/Variable
 *     Storage         | Zyada             | Kam
 *     Best For        | Read-heavy        | Write-heavy/high variance
 *     Celebrity Issue | Bada problem      | Problem nahi
*/

/**
 * ====================================================================
 * ROUND 5: HYBRID APPROACH - Yahan Interview Jeeta Jata Hai
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Achha, ab bata do, real-world mein Twitter jaisa system yeh
 *    kaise solve karta hai?"
 * 
 * CANDIDATE:
 *   "Twitter aur baaki bade systems HYBRID approach use karte hain.
 *    Core idea bahut simple hai:
 * 
 *      - NORMAL users (kam followers) -> unke liye Fanout-on-WRITE
 *        (push) - kyunki unke followers kam hain, load manageable hai
 * 
 *      - HIGH-FOLLOWER / CELEBRITY users -> unke liye Fanout-on-READ
 *        (pull) - kyunki push karne se system overload ho jayega
 * 
 *    Jab user apna feed maangta hai, hum DONO sources se data lete
 *    hain aur merge karte hain."
 * 
 *   DIAGRAM:
 * 
 *                     [New Post Aaya]
 *                           |
 *                           v
 *               Kya Author High-Follower hai?
 *                     /            \
 *                  NO /              \ YES
 *                    /                \
 *                   v                  v
 *          [Fanout-on-Write]      [Sirf DB mein Store karo]
 *                   |                  |
 *                   v                  v
 *          Follower Cache mein     [Posts DB]
 *          push kar do
 *                   |                  |
 *                   -------------------
 *                           |
 *                           v
 *                   [Feed Request Aaya]
 *                           |
 *                   Step 1: Cache se normal
 *                   authors ke posts lo
 *                           |
 *                   Step 2: High-follower authors
 *                   ke posts direct query karo
 *                           |
 *                   Step 3: Dono ko Merge karo
 *                   (time/score se)
 *                           |
 *                           v
 *                     [Final Feed]
 * 
 * INTERVIEWER:
 *   "High-follower threshold kaise decide karoge? Fixed number set
 *    kar doge?"
 * 
 * CANDIDATE:
 *   "Nahi sir, fixed number risky hai. Threshold TUNABLE hona chahiye
 *    aur yeh depend karta hai:
 *      - System ki fanout worker capacity kya hai
 *      - Author kitni frequency se post karta hai
 *      - Kitne followers ACTIVE hain (50K followers but sirf 1K
 *        active login karte hain - alag case hai 50K sab active se)
 * 
 *    Main conservative threshold se shuru karunga (jaise 10K
 *    followers), phir queue lag aur read latency monitor karke
 *    adjust karunga.
 * 
 *    Implementation ke liye, main user table mein ek flag rakhunga:
 * 
 *      user_A | 15M followers | eager_fanout = FALSE  (pull)
 *      user_B | 500 followers | eager_fanout = TRUE   (push)
 *      user_C | 50K followers | eager_fanout = FALSE  (pull)
 * 
 *    Ek background job periodically follower count check karke
 *    yeh flag update karta rahega. Aur main is flag ko FAST author
 *    metadata cache mein rakhunga, taaki post-creation ke time
 *    expensive graph query na karni pade."
*/

/**
 * ====================================================================
 * ROUND 6: EDGE CASES - Interviewer Yahan Depth Test Karega
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Achha ek scenario batata hoon - user B ne abhi User A ko follow
 *    kiya. Kya User B ko User A ke purane posts dikhenge?"
 * 
 * CANDIDATE:
 *   "Yeh depend karta hai model pe:
 *      - Fanout-on-Read mein: AUTOMATIC solve ho jata hai, kyunki
 *        read path already A ke recent posts query karta hai
 *      - Fanout-on-Write mein: PROBLEM hai, kyunki B ki feed cache
 *        purane follow se pehle bani thi, usme A ka data hi nahi hai
 * 
 *    Solution: Jab follow hota hai, main A ke recent posts ka ek
 *    BOUNDED window (jaise last 20 posts) B ki feed cache mein
 *    BACKFILL kar dunga. Sab kuch backfill karna zaroori nahi,
 *    kyunki users mostly recent content hi care karte hain."
 * 
 * INTERVIEWER:
 *   "Achha, agar koi post delete kar de jo already fanout ho chuka
 *    hai lakho logo ki feed mein, tab?"
 * 
 * CANDIDATE:
 *   "Cleanup problem hai. Approach yeh hoga:
 * 
 *     DIAGRAM:
 *       [Delete Request] -> [DB mein Mark Deleted]
 *                                 |
 *                                 v
 *                        [Async Cleanup Job]
 *                                 |
 *                                 v
 *                    [Feed Caches se Remove karo]
 * 
 *    Lekin IMPORTANT baat - main READ PATH pe bhi ek SAFETY CHECK
 *    rakhunga jo post ki visibility verify karega. Isse agar cleanup
 *    job abhi complete nahi hua, tab bhi user ko deleted post nahi
 *    dikhega."
 * 
 * INTERVIEWER:
 *   "Same logic block/mute/private account pe bhi lagega?"
 * 
 * CANDIDATE:
 *   "Bilkul sir. FEED CACHE FINAL SOURCE OF TRUTH nahi hai - yeh
 *    sirf ek PERFORMANCE OPTIMIZATION hai. Actual AUTHORIZATION
 *    check hamesha READ TIME pe hona chahiye. Agar B, A ko block
 *    karta hai, purani cached entries background cleanup se hategi,
 *    lekin CORRECTNESS us cleanup ke complete hone pe depend nahi
 *    karni chahiye - read path khud filter karega."
 * 
 * INTERVIEWER:
 *   "Ek aur - user post karta hai aur turant apna feed kholta hai,
 *    par usko apna hi post nahi dikhta. Kyun hota hai yeh, aur fix
 *    kaise karoge?"
 * 
 * CANDIDATE:
 *   "Yeh classic 'READ-YOUR-WRITES' problem hai. Aisa isliye hota
 *    hai kyunki fanout ABHI TAK async background mein chal raha
 *    hoga, ya read-model mein cache abhi update nahi hui.
 * 
 *    Teen tarike fix karne ke:
 *      1. Feed response banate waqt hamesha user ke OWN recent posts
 *         ko manually include kar do
 *      2. Client-side OPTIMISTIC UPDATE - UI mein turant dikha do,
 *         background mein confirm hone do
 *      3. Ek chhota AUTHOR-LOCAL TIMELINE rakho jo direct query hota
 *         hai feed assembly ke time, cache ke bharose na raho"
 * 
 * INTERVIEWER:
 *   "Ranked feed mein pagination kaise handle karoge? Kyunki naye
 *    posts continuously aa rahe hain."
 * 
 * CANDIDATE:
 *   "Main STABLE CURSOR use karunga (jaise timestamp + post_id ka
 *    combination), na ki simple OFFSET based pagination. Isse jab
 *    naye posts beech mein aate hain, toh existing pages mein
 *    duplicate ya missing posts ka issue nahi aayega."
*/

/**
 * ====================================================================
 * ROUND 7: SCALING & PERFORMANCE - Deep Technical Questions
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "10,000 followers ke liye individually 10,000 writes karoge?"
 * 
 * CANDIDATE:
 *   "Nahi sir, main BATCHING use karunga. Followers ko groups mein
 *    baant ke BULK operations karunga (jaise Redis pipeline ya
 *    batch insert), taaki network round-trips kam ho."
 * 
 * INTERVIEWER:
 *   "Post create hone ke baad user ko fanout complete hone tak wait
 *    karwaoge?"
 * 
 * CANDIDATE:
 *   "Bilkul nahi. Main ASYNC PROCESSING use karunga:
 * 
 *     DIAGRAM:
 *       [Post Service] --> [Message Queue] --> [Fanout Worker 1]
 *                                          --> [Fanout Worker 2]
 *                                          --> [Fanout Worker N]
 *                                                    |
 *                                                    v
 *                                             [Feed Cache]
 * 
 *    Post Service post save karke aur job queue mein daal ke TURANT
 *    user ko response de dega ('post ho gaya'). Fanout Workers
 *    background mein queue se consume karke feed caches update
 *    karenge. Isse:
 *      - Fast confirmation milta hai user ko
 *      - Workers HORIZONTALLY SCALE ho sakte hain
 *      - Failure pe RETRY ho sakta hai"
 * 
 * INTERVIEWER:
 *   "Agar worker crash ho jaye aur retry ho, toh duplicate entries
 *    ban sakti hain feed mein?"
 * 
 * CANDIDATE:
 *   "Haan, isliye workers IDEMPOTENT hone chahiye. Main
 *    (user_id, post_id) combination ko UNIQUE rakhunga feed cache
 *    mein, taaki agar same job do baar bhi process ho jaye, duplicate
 *    entry na bane."
 * 
 * INTERVIEWER:
 *   "Queue ko partition kaise karoge?"
 * 
 * CANDIDATE:
 *   "Depend karta hai priority pe:
 *      - Agar ek AUTHOR ke posts ka ORDER maintain karna zaroori hai,
 *        toh uske saare fanout jobs SAME PARTITION/ORDER mein rakhunga
 *      - Agar THROUGHPUT priority hai (jaise celebrity ka bada
 *        follower set), toh us set ko RECIPIENT-RANGES mein split
 *        kar dunga taaki multiple workers PARALLEL mein kaam kar sakein"
 * 
 * INTERVIEWER:
 *   "Follower list baar baar padhna expensive nahi hai kya?"
 * 
 * CANDIDATE:
 *   "Haan, isliye main follower/following lists ko bhi CACHE karunga
 *    (Redis Sets mein), aur BOUNDED PAGES mein padhunga. Isse graph
 *    database (jo shayad SQL ya graph DB hai) pe load nahi aayega
 *    har post/feed-request pe."
 * 
 * INTERVIEWER:
 *   "Poore system ka caching layer design karo."
 * 
 * CANDIDATE:
 *   "Multi-layer caching rakhunga:
 * 
 *     Layer        | Kaam                         | Technology
 *     -------------|------------------------------|------------------
 *     Feed cache   | Precomputed feed entries      | Redis Sorted Sets
 *     Post cache   | Hydrated post content          | Redis/Memcached
 *     User cache   | Follower lists, metadata        | Redis
 *     CDN          | Images, static content         | Cloudflare/CDN
 * 
 *    Aur feed cache ko SIZE-BOUND rakhunga - jaise sirf last 800
 *    posts ya 7 din ka data, purana DB se fetch hoga (rarely
 *    zaroorat padegi)."
 * 
 * INTERVIEWER:
 *   "Agar fanout queue bahut zyada backlog ho jaye (system overload),
 *    tab kya karoge?"
 * 
 * CANDIDATE:
 *   "Main BACKPRESSURE aur GRACEFUL DEGRADATION lagaunga:
 *      - Low-priority fanout jobs ko SLOW kar dunga
 *      - Notifications thodi der DELAY kar dunga
 *      - Candidate window CHHOTA kar dunga (kam posts consider karo)
 *      - Temporarily zyada PULL-based reads pe switch kar dunga
 * 
 *    MAIN PRIORITY yeh hai ki POST CREATION aur CORE FEED READS kabhi
 *    bhi impact na ho - fanout backlog ki wajah se poora system down
 *    nahi hona chahiye. Non-critical kaam pehle degrade hoga."
*/

/**
 * ====================================================================
 * ROUND 8: CLOSING - Interviewer Summary Maangega
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Theek hai, ek final summary do apne poore design ka."
 * 
 * CANDIDATE:
 *   "Sure sir. Meri final design yeh hai:
 * 
 *    1. NORMAL AUTHORS (kam followers) ke liye FANOUT-ON-WRITE
 *       (push model) - async queue + workers ke through, taaki
 *       unke followers ko fast, precomputed feed mile.
 * 
 *    2. HIGH-FOLLOWER AUTHORS (celebrities) ke liye FANOUT-ON-READ
 *       (pull model) - unke posts sirf DB mein store honge, read
 *       time pe fetch honge, taaki write amplification na ho.
 * 
 *    3. FEED GENERATION time pe, main dono sources - cache (normal
 *       authors) aur direct query (high-follower authors) - ko
 *       MERGE karunga.
 * 
 *    4. THRESHOLD tunable rakhunga aur monitor karke adjust karunga.
 * 
 *    5. EDGE CASES - delete, block, mute, new-follow, read-your-writes
 *       - sab ke liye READ-TIME SAFETY CHECKS lagaunga, kyunki cache
 *       final source of truth nahi hai.
 * 
 *    6. SCALE ke liye - batching, async queues, idempotent workers,
 *       queue partitioning, multi-layer caching, aur backpressure/
 *       degradation strategy use karunga.
 * 
 *    Yehi approach hai jo bade production feed systems (Twitter type)
 *    real mein use karte hain."
*/

/**
 * ====================================================================
 * RAPID FIRE REVISION (Last Minute Ke Liye)
 * ====================================================================
 * 
 * Q: Fanout kya hai?
 * A: Ek event -> many downstream updates. (1 post -> 1000s feed entries)
 * 
 * Q: Push model kise kehte hain?
 * A: Fanout-on-Write. Post likhte hi sabki feed cache update kar do.
 * 
 * Q: Pull model kise kehte hain?
 * A: Fanout-on-Read. Post sirf store karo, feed maangne par compute karo.
 * 
 * Q: Push model ki sabse badi problem?
 * A: High-follower author (celebrity) problem - millions of writes.
 * 
 * Q: Pull model ki sabse badi problem?
 * A: Read slow/unpredictable, especially jab user bahut logo ko follow
 *    karta hai.
 * 
 * Q: Real systems kya use karte hain?
 * A: HYBRID - normal users push, celebrities pull, feed time pe merge.
 * 
 * Q: Feed cache source of truth hai kya?
 * A: NAHI. Yeh sirf optimization hai. Visibility/auth check hamesha
 *    read-time pe hona chahiye (delete/block/mute ke liye).
 * 
 * Q: Read-your-writes problem kya hai?
 * A: User post karke turant apna post nahi dekh pata (fanout lag ki
 *    wajah se). Fix: apna post manually include karo response mein.
 * 
 * Q: Idempotency kyun zaroori hai fanout workers mein?
 * A: Kyunki retries duplicate feed entries bana sakte hain. Fix:
 *    (user_id, post_id) unique rakho.
 * 
 * Q: Data structure feed cache ke liye best kya hai?
 * A: Redis Sorted Set - post_id as member, timestamp as score.
 * 
 * Q: Naya follow karne pe purane posts kaise dikhaoge (push model mein)?
 * A: Bounded window backfill karo (jaise last 20 posts) follow hote
 *    hi.
 * 
 * Q: System overload mein kya priority rakhoge?
 * A: Post creation aur core feed reads protect karo, fanout/
 *    notifications ko degrade karo pehle.
 * 
 * Note: Practice bolke karo, whiteboard pe diagram banane ki practice
 *       bhi karo - interview mein yehi flow follow karna hai.
*/